# Lecture 7: Statistical Modelling Part II

:::{admonition} Learning Objectives
:class: tip
After this lecture, you will be able to:
- Evaluate model accuracy beyond simple metrics (bias, deviance, ranking)
- Apply regularisation (L1, L2) to control model complexity
- Perform feature selection to reduce dimensionality
- Tune hyperparameters systematically (grid search, random search, Bayesian)
- Implement advanced modelling techniques: interaction/monotonicity constraints, custom loss functions
:::

## 1. Model Accuracy

Multiple dimensions of model quality:

- **Bias**: Are predicted means correct? (compare weighted mean predictions with weighted mean outcomes)
- **Validation error**: Deviance, MSE, MAE, MSPE
- **Ranking of predictions**: How well does the model sort observations? (Normalized Gini / CAP curve)

### Deviance

A generalisation of squared residuals for maximum likelihood estimation (GLMs):

$$D(y, \hat{\mu}) = 2\left(\log p(y|\hat{\theta}_s) - \log p(y|\hat{\theta}_0)\right)$$

where $\hat{\theta}_s$ are the parameters of the saturated (perfect) model.
The lower the deviance, the better.

### Normalized Gini (CAP Curve)

Measures how well the model ranks observations. Adapted from the Lorenz curve in economics:

$$\text{Gini}_{\text{ML}} = \frac{A}{A + B}$$

where the Cumulative Accuracy Profile (CAP) shows what fraction of the total target is captured by the top-ranked predictions.

In [1]:
# TODO: Predicted vs Actual plot
# from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# from sklearn.linear_model import Lasso
# from fun_ds.plotting import plot_residuals, set_lecture_style
#
# set_lecture_style()
# # Compare multiple models visually
# # Plot predicted vs actual for each model

## 2. Regularisation

Regularisation prevents overfitting by penalising model complexity.

| Method | Penalty | Effect |
|--------|---------|--------|
| Ridge (L2) | $\lambda \sum \beta_j^2$ | Shrinks all coefficients |
| Lasso (L1) | $\lambda \sum |\beta_j|$ | Drives some coefficients to zero |
| Elastic Net | $\alpha \lambda \sum|\beta_j| + \frac{(1-\alpha)}{2}\lambda \sum\beta_j^2$ | Combines both |

In [2]:
# TODO: Regularisation path visualisation
# from sklearn.linear_model import Ridge, Lasso, ElasticNet
# import numpy as np
#
# # Show how coefficients change with regularisation strength
# alphas = np.logspace(-3, 3, 50)
# coefs_ridge = []
# for alpha in alphas:
#     model = Ridge(alpha=alpha).fit(X_train, y_train)
#     coefs_ridge.append(model.coef_)
#
# # Plot regularisation paths

## 3. Feature Selection

Methods to reduce the number of features:

1. **Filter methods**: Correlation, mutual information, variance threshold
2. **Wrapper methods**: Forward/backward selection, recursive feature elimination
3. **Embedded methods**: L1 regularisation (built into the model)

:::{note}
Feature selection is less critical for tree-based models which handle irrelevant features naturally, but it improves interpretability and can reduce overfitting for linear models.
:::

In [3]:
# TODO: Feature selection example
# from sklearn.feature_selection import (
#     SelectKBest, mutual_info_regression, RFECV
# )
#
# # Filter: mutual information
# selector = SelectKBest(mutual_info_regression, k=5)
# X_selected = selector.fit_transform(X_train, y_train)
#
# # Wrapper: recursive feature elimination with CV
# rfecv = RFECV(Ridge(alpha=1.0), step=1, cv=5)
# rfecv.fit(X_train, y_train)

## 4. Hyperparameter Tuning

Hyperparameters control model complexity and must be set before training.

| Strategy | Pros | Cons |
|----------|------|------|
| Grid Search | Exhaustive | Exponential cost |
| Random Search | Efficient in high dimensions | Less systematic |
| Bayesian (Optuna) | Sample-efficient | More complex setup |

In [4]:
# TODO: Hyperparameter tuning example
# from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
#
# param_grid = {
#     "alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
# }
#
# grid_search = GridSearchCV(
#     Ridge(), param_grid, cv=5, scoring='r2'
# )
# grid_search.fit(X_train, y_train)
#
# print(f"Best alpha: {grid_search.best_params_}")
# print(f"Best R2: {grid_search.best_score_:.4f}")

## 5. Advanced Modelling Topics

### Interaction constraints

In gradient boosting, we can restrict which features may interact in the same tree. This encodes domain knowledge (e.g., geographic features should not interact with temporal features).

### Monotonicity constraints

Force the model to respect known monotonic relationships:
- Higher income $\rightarrow$ higher house value
- More experience $\rightarrow$ higher salary

Supported natively in LightGBM and XGBoost via `monotone_constraints`.

### Custom loss functions in LightGBM

When built-in loss functions don't match your objective, implement your own:
- Provide gradient and hessian functions
- Examples: asymmetric loss, business-specific objectives

In [5]:
# TODO: Custom loss function example
# import lightgbm as lgb
#
# def asymmetric_mse(y_true, y_pred):
#     """Penalise under-prediction 2x more than over-prediction."""
#     residual = y_true - y_pred
#     grad = np.where(residual > 0, -2 * residual, -residual)
#     hess = np.where(residual > 0, 2.0, 1.0)
#     return grad, hess
#
# model = lgb.LGBMRegressor(objective=asymmetric_mse)
# model.fit(X_train, y_train)

:::{admonition} Key Takeaways
:class: important
- Evaluate models on multiple dimensions: bias, validation error, and ranking
- Regularisation (L1/L2) controls complexity — tune the strength via CV
- Feature selection improves interpretability and can reduce overfitting
- Use random search or Bayesian optimisation over grid search for efficiency
- Modern GBMs support constraints (monotonicity, interactions) and custom losses
:::